# AI Lens: Plant Leaf Disease Detection
### MobileNetV2 Transfer Learning on PlantVillage (38 Classes, 14 Crop Species)

This notebook trains the deep learning model for the **AI Lens** college project.

**Key Highlights:**
- **Dataset**: Official PlantVillage repository (`mohanty/PlantVillage` on Hugging Face, color configuration)
- **Compatibility**: Directly downloads the official dataset archive from Hugging Face, bypassing deprecated `trust_remote_code` script requirements in modern `datasets` versions
- **Architecture**: MobileNetV2 pretrained on ImageNet
- **Preprocessing**: Canonical MobileNetV2 normalization $[-1, 1]$ at $224 \times 224$
- **Regularization**: Data augmentation on training images only, Dropout, Batch Normalization
- **Optimization**: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
- **Evaluation**: Accuracy, Macro Precision, Recall, F1-Score, and Confusion Matrix
- **Artifacts**: Exports `plant_disease_mobilenetv2.keras` and `class_names.json` for the Streamlit web application.

## 1. Verify GPU Hardware Acceleration
In Google Colab, select: **Runtime** -> **Change runtime type** -> **T4 GPU**.

In [ ]:
!nvidia-smi
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("Available GPUs:", gpus)
assert len(gpus) > 0, "No GPU detected! Please enable GPU accelerator in Runtime settings."

## 2. Install Required Packages

In [ ]:
!pip install -q scikit-learn seaborn matplotlib

## 3. Download Official PlantVillage Archive from Hugging Face
We download the official `data.zip` directly from the `mohanty/PlantVillage` Hugging Face repository and extract the 38 color leaf classes.

In [ ]:
import os
import zipfile

DATASET_URL = "https://huggingface.co/datasets/mohanty/PlantVillage/resolve/main/data.zip"
ZIP_PATH = "plantvillage_data.zip"
TARGET_DIR = "plantvillage_color"

# Download official data.zip from Hugging Face
if not os.path.exists(ZIP_PATH):
    print("Downloading official PlantVillage archive from Hugging Face...")
    !wget -c https://huggingface.co/datasets/mohanty/PlantVillage/resolve/main/data.zip -O plantvillage_data.zip

# Extract the 38 color class folders
if not os.path.exists(TARGET_DIR) or len(os.listdir(TARGET_DIR)) < 38:
    print("Extracting color images into target directory...")
    os.makedirs(TARGET_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        color_members = [m for m in zf.namelist() if m.startswith("raw/color/") and not m.endswith("/")]
        print(f"Unpacking {len(color_members)} images across 38 classes...")
        for member in color_members:
            rel_path = member[len("raw/color/"):]
            out_path = os.path.join(TARGET_DIR, rel_path)
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with zf.open(member) as src, open(out_path, "wb") as dst:
                dst.write(src.read())
    print("Extraction complete!")
else:
    print("Dataset already extracted and ready!")

## 4. Load Dataset & Create Disjoint Train / Val / Test Splits (70% / 15% / 15%)

In [ ]:
import json

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# Load full dataset with TensorFlow
full_ds = tf.keras.utils.image_dataset_from_directory(
    TARGET_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

label_names = full_ds.class_names
print(f"Total Verified Classes: {len(label_names)}")
print("Sample Classes:", label_names[:5])

# Export canonical class names for the web application
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
with open("models/class_names.json", "w") as f:
    json.dump(label_names, f, indent=2)
print("Saved models/class_names.json")

# Partition into Train (70%), Validation (15%), and Test (15%)
total_batches = len(full_ds)
test_batches = max(1, int(total_batches * 0.15))
val_batches = max(1, int(total_batches * 0.15))
train_batches = total_batches - test_batches - val_batches

test_ds = full_ds.take(test_batches)
remaining_ds = full_ds.skip(test_batches)
val_ds = remaining_ds.take(val_batches)
train_ds = remaining_ds.skip(val_batches)

print(f"Batches: Total={total_batches} | Train={train_batches} | Val={val_batches} | Test={test_batches}")

## 5. High-Performance Input Pipelines with Training-Only Data Augmentation

In [ ]:
# Data Augmentation applied ONLY to training data
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="training_augmentation")

train_ds = (
    train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

## 6. Build MobileNetV2 Transfer Learning Architecture

In [ ]:
from tensorflow.keras import layers, Model

inputs = layers.Input(shape=(224, 224, 3), name="leaf_input")

# MobileNetV2 normalization: maps [0, 255] -> [-1.0, 1.0]
x = layers.Rescaling(scale=1./127.5, offset=-1.0, name="mobilenetv2_rescaling")(inputs)

# Pretrained base model (ImageNet weights)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze base during phase 1

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.BatchNormalization(name="bn1")(x)
x = layers.Dropout(0.3, name="drop1")(x)
x = layers.Dense(256, activation="relu", name="dense_head")(x)
x = layers.BatchNormalization(name="bn2")(x)
x = layers.Dropout(0.2, name="drop2")(x)
outputs = layers.Dense(len(label_names), activation="softmax", name="probabilities")(x)

model = Model(inputs=inputs, outputs=outputs, name="AI_Lens_MobileNetV2")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 7. Train Phase 1 (Frozen Base) with EarlyStopping & Checkpointing

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="models/plant_disease_mobilenetv2.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

EPOCHS = 12
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 8. Train Phase 2 (Fine-Tuning Top Layers)

In [ ]:
# Unfreeze top layers for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

FINE_TUNE_EPOCHS = 5
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks
)

## 9. Comprehensive Model Evaluation on Separate Test Set

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

y_true_all = []
y_pred_all = []

for imgs, lbls in test_ds:
    p = model.predict(imgs, verbose=0)
    y_true_all.extend(lbls.numpy())
    y_pred_all.extend(np.argmax(p, axis=1))

y_true = np.array(y_true_all)
y_pred = np.array(y_pred_all)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec = recall_score(y_true, y_pred, average="macro", zero_division=0)

print(f"\nTest Accuracy:  {acc * 100:.2f}%")
print(f"Test Macro F1:  {f1 * 100:.2f}%")
print(f"Test Precision: {prec * 100:.2f}%")
print(f"Test Recall:    {rec * 100:.2f}%")

print("\nClassification Report:")
cr = classification_report(y_true, y_pred, target_names=label_names, digits=4, zero_division=0)
print(cr)

with open("reports/classification_report.txt", "w") as f:
    f.write(cr)

## 10. Confusion Matrix Plot

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap="Greens", annot=False, xticklabels=[c[:12] for c in label_names], yticklabels=[c[:12] for c in label_names])
plt.title("AI Lens: PlantVillage 38-Class Confusion Matrix (MobileNetV2)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("reports/confusion_matrix.png", dpi=300)
plt.show()

## 11. Download Artifacts for Local Web App

In [ ]:
from google.colab import files

print("Downloading trained model weights and class names...")
files.download("models/plant_disease_mobilenetv2.keras")
files.download("models/class_names.json")
print("Download initiated! Place these files into your local AI-Lens/models/ directory.")